# Quantitative Analysis — Milei Climate Narratives Corpus (v4)

Reproducible notebook for the analysis of `milei_corpus_v4_code_counts.csv` (193 coded speeches, Dimension 1 codebook v4).

**What this notebook does, in order:**
1. Data prep & QA — load the corpus, compute word counts, build a long-format code table
2. Descriptive overview — code distributions, sub-code ranking, zero-code speech profile
3. Time trend — code density by quarter across the presidency
4. Venue & country effects — cross-tabs by `tipo`/`pais` with Kruskal-Wallis tests
5. Climate-focused deep dive — validating the `clima` heuristic, "climate-signature" sub-codes, top speeches
6. Co-occurrence & rhetorical-mode clustering — sub-code correlation matrix + k-means clusters

**Requirements:** `pandas`, `numpy`, `scipy`, `scikit-learn`, `matplotlib`. Install with:
```
pip install pandas numpy scipy scikit-learn matplotlib
```

**Paths:** set `BASE_DIR` below to the root of the `milei-climate-narratives` project folder (the folder containing `analysis/` and `data/`). Everything else is relative to it.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib
matplotlib.use('Agg')  # switch to default backend if running interactively, e.g. remove this line in Jupyter
import matplotlib.pyplot as plt

# --- EDIT THIS to point at your local project folder ---
BASE_DIR = "."  # e.g. r"C:\Users\you\milei-climate-narratives"

CORPUS_CSV = os.path.join(BASE_DIR, "analysis/results/v4/milei_corpus_v4_code_counts.csv")
RAW_SPEECHES_DIR = os.path.join(BASE_DIR, "data/raw_speeches")
OUT_DIR = os.path.join(BASE_DIR, "analysis/results/v4/quant_analysis")
CHARTS_DIR = os.path.join(OUT_DIR, "charts")
os.makedirs(CHARTS_DIR, exist_ok=True)

pd.set_option('display.width', 140)
pd.set_option('display.max_rows', 40)

## 1. Data prep & QA

Load the master code-count CSV, confirm all rows are `coded=ok`, and compute a **word count per speech** from the raw (uncoded) transcripts in `data/raw_speeches/` — using the raw files rather than the annotated ones in `analysis/coded_speeches/v4/` avoids inflating counts with `[CODE: ...]` tags. Word counts let us normalize code counts by speech length later (a 300-word cadena nacional clip and a 7,700-word Davos keynote are not comparable on raw counts alone).

In [ ]:
df = pd.read_csv(CORPUS_CSV)
print("Rows:", len(df), "| Columns:", len(df.columns))
print("coded == 'ok' for all rows:", (df['coded'] == 'ok').all())

subcodes = [
    'D1.1_Market_Primacy', 'D1.1_Anti_State_Anti_Bureaucratic', 'D1.1_Anti_Casta',
    'D1.1_Delegitimization_of_Regulation', 'D1.1_Sustainability_as_Ideological_or_Harmful',
    'D1.1_Rejection_of_Multilateral_Governance',
    'D1.2_Attack_on_Universities_and_Research_Institutions', 'D1.2_Attack_on_Experts_and_Technocrats',
    'D1.2_Climate_Skepticism_or_Dismissal', 'D1.2_Pseudo_Science_or_Ideological_Science', 'D1.2_Selective_Empiricism',
    'D1.3_Woke_Bundling', 'D1.3_Anti_Socialism_Anti_Collectivism', 'D1.3_Civilizational_Defense',
]

In [ ]:
def word_count(fname):
    """Word count from the raw (unannotated) transcript matched via `matched_coded_file`."""
    path = os.path.join(RAW_SPEECHES_DIR, fname)
    if not os.path.exists(path):
        return None
    txt = open(path, encoding='utf-8').read()
    # strip YAML frontmatter (content between the first two '---' markers)
    parts = txt.split('---')
    body = '---'.join(parts[2:]) if len(parts) >= 3 else txt
    return len(re.findall(r"\b\w+\b", body, re.UNICODE))

df['word_count'] = df['matched_coded_file'].apply(word_count)
print("Missing word counts:", df['word_count'].isna().sum())
df['word_count'].describe()

In [ ]:
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
print("Date parse failures:", df['fecha'].isna().sum())
print("Date range:", df['fecha'].min().date(), "to", df['fecha'].max().date())

df['year'] = df['fecha'].dt.year
df['quarter'] = df['fecha'].dt.to_period('Q').astype(str)
df['month'] = df['fecha'].dt.to_period('M').astype(str)

# Rates per 1,000 words -- the primary normalized metric used throughout this notebook
df['codes_per_1k'] = df['Codes_Total'] / df['word_count'] * 1000
df['d11_per_1k'] = df['D1.1_Total'] / df['word_count'] * 1000
df['d12_per_1k'] = df['D1.2_Total'] / df['word_count'] * 1000
df['d13_per_1k'] = df['D1.3_Total'] / df['word_count'] * 1000

df.to_csv(os.path.join(OUT_DIR, 'corpus_enriched.csv'), index=False)
df[['fecha', 'quarter', 'word_count', 'Codes_Total', 'codes_per_1k']].head()

Build a **long-format table** (one row per code instance) alongside the wide table — this makes sub-code ranking and co-occurrence analysis much simpler than pivoting the wide table each time.

In [ ]:
long = df.melt(
    id_vars=['id', 'fecha', 'year', 'quarter', 'pais', 'tipo', 'subtipo', 'clima', 'word_count'],
    value_vars=subcodes, var_name='subcode', value_name='count'
)
long = long[long['count'] > 0].copy()
long['dimension'] = long['subcode'].str.split('_').str[0]
long.to_csv(os.path.join(OUT_DIR, 'long_format.csv'), index=False)
print("Long-format rows (code instances):", len(long))

## 2. Descriptive overview

Distribution of total codes per speech, ranking of the 14 sub-codes by frequency, and a profile of the zero-code speeches (do they cluster in a particular genre, or are they scattered randomly?).

In [ ]:
print(df['Codes_Total'].describe())
print()
dim_totals = df[['D1.1_Total', 'D1.2_Total', 'D1.3_Total']].sum()
print("Dimension totals:")
print(dim_totals)
print((dim_totals / dim_totals.sum() * 100).round(1), "% share")

In [ ]:
sub_totals = long.groupby('subcode')['count'].sum().sort_values(ascending=False)
sub_totals

In [ ]:
zero = df[df['Codes_Total'] == 0]
print("Zero-code speeches:", len(zero), f"({len(zero)/len(df)*100:.0f}% of corpus)")
print()
print("By tipo:")
print(zero['tipo'].value_counts())
print()
print("By pais:")
print(zero['pais'].value_counts())
print()
print("By clima flag:")
print(zero['clima'].fillna('blank').value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['Codes_Total'], bins=range(0, 21), color='#4A4A4A', edgecolor='white')
ax.set_xlabel('Codes_Total per speech')
ax.set_ylabel('Number of speeches')
ax.set_title(f"Distribution of D1 code counts per speech (n={len(df)}, {(df.Codes_Total==0).sum()} zero-code)")
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '03_codes_histogram.png'), dpi=150)
plt.show()

In [ ]:
short_names = [c.split('_', 1)[1].replace('_', ' ') for c in sub_totals.index]
colors = ['#8B2E2E' if s.startswith('D1.1') else '#2E6B4A' if s.startswith('D1.2') else '#2E4A8B'
          for s in sub_totals.index]

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(short_names[::-1], sub_totals.values[::-1], color=colors[::-1])
ax.set_xlabel('Total instances across corpus')
ax.set_title('D1 sub-code frequency ranking\n(red=D1.1 State/Sustainability, green=D1.2 Science/Expertise, blue=D1.3 Cultural Backlash)')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '02_subcode_ranking.png'), dpi=150)
plt.show()

## 3. Time trend

Aggregate by quarter and compute **code density** (codes per 1,000 words), not just raw code volume, since the number and length of speeches per quarter varies a lot.

In [ ]:
q = df.groupby('quarter').agg(
    n_speeches=('id', 'count'),
    total_codes=('Codes_Total', 'sum'),
    total_words=('word_count', 'sum'),
).reset_index()
q['codes_per_1k'] = q['total_codes'] / q['total_words'] * 1000
q

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].bar(q['quarter'], q['total_codes'], color='#8B2E2E')
axes[0].set_ylabel('Total D1 codes')
axes[0].set_title('D1 code volume by quarter (raw count)')
axes[0].tick_params(axis='x', rotation=90)

axes[1].plot(q['quarter'], q['codes_per_1k'], marker='o', color='#2E4A8B')
axes[1].set_ylabel('Codes per 1,000 words')
axes[1].set_title('D1 code density by quarter (normalized by word count)')
axes[1].tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '01_time_series.png'), dpi=150)
plt.show()

## 4. Venue & country effects

Cross-tabulate code density (`codes_per_1k`) by `tipo` (venue/speech type) and `pais` (country), and test for group differences with the **Kruskal-Wallis** test (nonparametric — the rate distribution is right-skewed, so ANOVA's normality assumption doesn't hold). Groups below a minimum size are excluded from the test itself since a single-speech "group" doesn't provide a meaningful test of central tendency.

In [ ]:
tipo_stats = df.groupby('tipo').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    median_codes_per_1k=('codes_per_1k', 'median'),
    mean_word_count=('word_count', 'mean'),
).sort_values('mean_codes_per_1k', ascending=False)
tipo_stats.to_csv(os.path.join(OUT_DIR, 'tipo_stats.csv'))
tipo_stats.round(2)

In [ ]:
# Kruskal-Wallis across tipo groups with n >= 5 (stable estimate threshold)
groups_tipo = [g['codes_per_1k'].values for name, g in df.groupby('tipo') if len(g) >= 5]
h, p = stats.kruskal(*groups_tipo)
print(f"Kruskal-Wallis across tipo (k={len(groups_tipo)} groups with n>=5): H={h:.2f}, p={p:.4f}")

In [ ]:
pais_stats = df.groupby('pais').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    median_codes_per_1k=('codes_per_1k', 'median'),
).sort_values('mean_codes_per_1k', ascending=False)
pais_stats.to_csv(os.path.join(OUT_DIR, 'pais_stats.csv'))
pais_stats.round(2)

In [ ]:
groups_pais = [g['codes_per_1k'].values for name, g in df.groupby('pais') if len(g) >= 3]
h2, p2 = stats.kruskal(*groups_pais)
print(f"Kruskal-Wallis across pais (k={len(groups_pais)} groups with n>=3): H={h2:.2f}, p={p2:.4f}")

# Argentina vs. rest-of-world binary comparison
df['ar_vs_intl'] = np.where(df['pais'] == 'argentina', 'argentina', 'international')
u, pu = stats.mannwhitneyu(
    df.loc[df.ar_vs_intl == 'argentina', 'codes_per_1k'],
    df.loc[df.ar_vs_intl == 'international', 'codes_per_1k'],
)
print(df.groupby('ar_vs_intl')['codes_per_1k'].agg(['count', 'mean', 'median']).round(3))
print(f"Mann-Whitney U={u:.1f}, p={pu:.4f}")

## 5. Climate-focused deep dive

Two questions here:

1. **Does the corpus's `clima` selection heuristic (✔ / ~ / blank) actually predict code density?** If yes, that validates using it as a speech-selection filter.
2. **Does climate-obstruction rhetoric bleed into speeches *not* flagged as climate-relevant?** Defined via three "climate-signature" sub-codes: `Sustainability_as_Ideological_or_Harmful`, `Rejection_of_Multilateral_Governance`, `Climate_Skepticism_or_Dismissal`. If a large share of these instances falls outside `clima=✔` speeches, that supports coding climate-adjacent rhetoric wherever it appears rather than gating on an explicit climate mention per passage.

In [ ]:
df['clima_f'] = df['clima'].fillna('blank')

profile = df.groupby('clima_f').agg(
    n=('id', 'count'),
    mean_codes_per_1k=('codes_per_1k', 'mean'),
    mean_d11=('d11_per_1k', 'mean'),
    mean_d12=('d12_per_1k', 'mean'),
    mean_d13=('d13_per_1k', 'mean'),
)
print(profile.round(3))

groups_clima = [g['codes_per_1k'].values for name, g in df.groupby('clima_f')]
h3, p3 = stats.kruskal(*groups_clima)
print(f"\nKruskal-Wallis across clima groups: H={h3:.2f}, p={p3:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
order = ['blank', '~', '✔']
vals = [df.loc[df.clima_f == c, 'codes_per_1k'].mean() for c in order]
labels = ['No climate\nrelevance flagged', 'Possible climate\nrelevance (~)', 'High climate\nrelevance (✔)']
ax.bar(labels, vals, color=['#999999', '#D9A441', '#8B2E2E'])
ax.set_ylabel('Mean D1 codes per 1,000 words')
ax.set_title(f'D1 code density by climate-relevance heuristic (Kruskal-Wallis p={p3:.4f})')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '04_clima_density.png'), dpi=150)
plt.show()

In [ ]:
sig_cols = [
    'D1.1_Sustainability_as_Ideological_or_Harmful',
    'D1.1_Rejection_of_Multilateral_Governance',
    'D1.2_Climate_Skepticism_or_Dismissal',
]
df['climate_signature_total'] = df[sig_cols].sum(axis=1)
df['climate_signature_per_1k'] = df['climate_signature_total'] / df['word_count'] * 1000

inside = df.loc[df.clima_f == '✔', 'climate_signature_total'].sum()
outside = df.loc[df.clima_f != '✔', 'climate_signature_total'].sum()
total = inside + outside
print(f"Climate-signature instances inside clima=✔ speeches: {inside} ({inside/total*100:.0f}%)")
print(f"Climate-signature instances outside clima=✔ speeches: {outside} ({outside/total*100:.0f}%)")
print(f"Total: {total}")

df.to_csv(os.path.join(OUT_DIR, 'corpus_enriched.csv'), index=False)  # re-save with the new columns

In [ ]:
top = (
    df[df['climate_signature_total'] > 0]
    .sort_values('climate_signature_per_1k', ascending=False)
    [['fecha', 'evento-discurso', 'pais', 'tipo', 'clima_f', 'climate_signature_total', 'word_count', 'climate_signature_per_1k']]
    .head(15)
)
top

In [ ]:
q_sig = df.groupby('quarter').agg(sig_sum=('climate_signature_total', 'sum')).reset_index()
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(q_sig['quarter'], q_sig['sig_sum'], color='#2E6B4A')
ax.set_ylabel('Climate-signature code instances')
ax.set_title('Climate-signature codes (Sustainability_as_Ideological, Rejection_of_Multilateral_Gov, Climate_Skepticism) by quarter')
ax.tick_params(axis='x', rotation=90)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '05_climate_signature_time.png'), dpi=150)
plt.show()

## 6. Co-occurrence and rhetorical-mode clustering

**Co-occurrence:** phi coefficients (Pearson correlation on binary presence/absence) across the 14 sub-codes — which codes tend to appear together within the same speech?

**Clustering:** k-means on each speech's code-**composition** profile (i.e. what % of that speech's codes belong to each sub-code — the "shape" of its rhetorical mix, independent of how many codes it has overall). Composition is used instead of raw rate-per-word because very short, high-rate outlier speeches (e.g. a 300-word debate clip) otherwise dominate a rate-based clustering. Only speeches with `Codes_Total >= 3` are included so each profile is estimated from enough data points to be stable.

In [ ]:
X_bin = (df[subcodes] > 0).astype(int)
corr = X_bin.corr()

fig, ax = plt.subplots(figsize=(11, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(short_names))); ax.set_xticklabels(short_names, rotation=90)
ax.set_yticks(range(len(short_names))); ax.set_yticklabels(short_names)
plt.colorbar(im, label='Phi coefficient (co-occurrence correlation)')
ax.set_title(f'Sub-code co-occurrence correlation (presence/absence per speech, n={len(df)})')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '06_cooccurrence_heatmap.png'), dpi=150)
plt.show()

In [ ]:
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print("Top 10 co-occurring sub-code pairs:")
pairs.head(10)

In [ ]:
sub = df[df['Codes_Total'] >= 3].copy()
comp = sub[subcodes].div(sub['Codes_Total'], axis=0)  # composition: each subcode's share of that speech's total codes
X = StandardScaler().fit_transform(comp)

print(f"n speeches with Codes_Total>=3: {len(sub)}\n")
for k in range(2, 7):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    score = silhouette_score(X, km_k.labels_)
    print(f"k={k}: silhouette={score:.3f}, cluster sizes={np.bincount(km_k.labels_)}")

In [ ]:
# k=4 chosen: interpretable clusters at a reasonable silhouette score for compositional political-rhetoric data
K = 4
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
sub['cluster'] = km.labels_

for c in sorted(sub['cluster'].unique()):
    grp = sub[sub['cluster'] == c]
    print(f"--- Cluster {c} (n={len(grp)}, mean codes/1k={grp['codes_per_1k'].mean():.2f}) ---")
    top_codes = comp.loc[grp.index].mean().sort_values(ascending=False).head(4)
    print((top_codes * 100).round(1).astype(str) + '%')
    print('tipo top3:', grp['tipo'].value_counts().head(3).to_dict())
    print('pais top3:', grp['pais'].value_counts().head(3).to_dict())
    print()

sub.to_csv(os.path.join(OUT_DIR, 'clustered_speeches.csv'), index=False)

In [ ]:
pca = PCA(n_components=2).fit(X)
coords = pca.transform(X)
sub['pc1'], sub['pc2'] = coords[:, 0], coords[:, 1]

cluster_labels = {0: 'Cluster 0', 1: 'Cluster 1', 2: 'Cluster 2', 3: 'Cluster 3'}  # relabel descriptively after inspecting means above
colors = {0: '#8B2E2E', 1: '#2E4A8B', 2: '#2E6B4A', 3: '#999999'}

fig, ax = plt.subplots(figsize=(9, 7))
for c in sorted(sub['cluster'].unique()):
    grp = sub[sub['cluster'] == c]
    ax.scatter(grp['pc1'], grp['pc2'], label=f"{cluster_labels[c]} (n={len(grp)})", color=colors[c], s=60, alpha=0.8)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.0f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.0f}% var)')
ax.set_title(f'Rhetorical-mode clusters by D1 code composition (Codes_Total>=3, n={len(sub)})')
ax.legend(fontsize=8, loc='best')
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, '07_cluster_pca.png'), dpi=150)
plt.show()

## Done

All intermediate tables (`corpus_enriched.csv`, `long_format.csv`, `tipo_stats.csv`, `pais_stats.csv`, `clustered_speeches.csv`) and all 7 charts are written to `analysis/results/v4/quant_analysis/`. See `quant_analysis_memo.md` in the same folder for the full write-up of findings and their implications for the paper draft.